In [15]:
import rasterio
import numpy as np
import joblib # Standard for loading sklearn models
import pandas as pd

vars = ["Aridity_Index", "ET", "GPP", "LST", "NDVI", "Slope", "TWI", "bio01", "bio02", "bio03", "bio04", "bio05", "bio06", "bio07", "bio08", "bio09", "bio10", "bio11", "bio12", "bio13", "bio14", "bio15", "bio16", "bio17", "bio18", "bio19"]

# 1. Setup paths
raster_path = "/home/kmisbah/CESBIO/AL_Moutmir/alMoutmir_Modeling/inference/Ain_Omra_covariates_30m.tif" # Must contain 26 bands matching the exact order of 'vars'
model_path = "/home/kmisbah/CESBIO/AL_Moutmir/alMoutmir_Modeling/mlruns/420775402803154173/models/m-9b9f0cc098094df2adc0330d33c9fe90/artifacts/model.pkl"
output_path = "prediction_p2o5_ehangable_olsen_ppm_30m.tif"

# 2. Load the trained scikit-learn model
model = joblib.load(model_path)

# 3. Read the raster data
with rasterio.open(raster_path) as src:
    meta = src.meta.copy()
    nodata_val = src.nodata
    
    # Check if the raster has band names saved
    saved_band_names = src.descriptions
    
    # If the TIFF has band names, find the exact indices we need
    if any(saved_band_names):
        # Rasterio is 1-indexed for reading specific bands, so we add 1
        band_indices = [saved_band_names.index(var) + 1 for var in vars]
        # Read ONLY the 26 bands we care about
        img_data = src.read(band_indices)
    else:
        # Fallback if no names are saved (reads everything)
        img_data = src.read()

bands, rows, cols = img_data.shape
print(f"Loaded {bands} bands.") # This should now print 26!

# 4. Reshape for scikit-learn
# Transpose to (rows, cols, bands), then flatten to (rows * cols, bands)
img_data_2d = img_data.transpose(1, 2, 0).reshape(-1, bands)

# 5. Mask out NoData/NaN values
# Sklearn will throw an error if it encounters NaNs or extreme nodata values
if nodata_val is not None:
    # Keep pixels where NOT ALL bands equal the nodata value
    valid_mask = ~np.any(img_data_2d == nodata_val, axis=1)
else:
    # Fallback: check for standard NaNs
    valid_mask = ~np.isnan(img_data_2d).any(axis=1)

# Extract only the valid pixels for inference
valid_pixels = img_data_2d[valid_mask]

# 6. Run the spatial prediction
# Create an empty array for predictions filled with a new NoData value (e.g., -9999)
predictions_1d = np.full(rows * cols, -9999, dtype=np.float32)

# 6. Run the spatial prediction
# Create an empty array for predictions filled with a new NoData value (e.g., -9999)
predictions_1d = np.full(rows * cols, -9999, dtype=np.float32)

if valid_pixels.shape[0] > 0:
    # --- THE FIX IS HERE ---
    # Wrap the valid numpy pixels in a DataFrame using your 'vars' list
    valid_pixels_df = pd.DataFrame(valid_pixels, columns=vars)
    valid_pixels_df['SU_WRB1_PH'] = 'KS'
    
    # Predict using the DataFrame
    pred = model.predict(valid_pixels_df)
    
    # Map results back to valid pixel locations
    predictions_1d[valid_mask] = pred

# 7. Reshape back to spatial dimensions
prediction_2d = predictions_1d.reshape(rows, cols)

# 8. Export the predicted raster
# Update metadata for a single-band output
meta.update({
    "count": 1,
    "dtype": "float32", # Use 'int32' if this is a classification model
    "nodata": -9999
})

with rasterio.open(output_path, 'w', **meta) as dst:
    dst.write(prediction_2d, 1)

print("Spatial prediction complete and saved.")

Loaded 26 bands.
Spatial prediction complete and saved.
